<a href="https://colab.research.google.com/github/YuYing33/CS521_FA26_HW/blob/main/P2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 521 HW4, Problem 2: Certified Training for Multiple Norms (CURE)

Open this notebook in Google Colab and select a GPU runtime (*Runtime > Change runtime type > T4 GPU*).

Complete each part that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the verifier, the evaluation or the default hyperparameters.
Change them only if the homework tells you to.

**Threat model.** Let $f$ be the network and $x$ an image with the label $y$. The network is *certified* at $x$
for the norm $p$ if the verifier proves this: $f_y(x') - f_j(x') > 0$ for all $j \neq y$ and all $x'$ with
$\|x' - x\|_p \leq \epsilon_p$. For $\ell_\infty$, $x'$ must also be in $[0, 1]^{784}$.
We use $\epsilon_\infty = 0.1$ and $\epsilon_2 = 1.0$.
The *union certified accuracy* is the percentage of test images where the network is certified for the two norms.

In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

EPS = {"linf": 0.1, "l2": 1.0}     # the radii of the threat model
NORMS = ["linf", "l2"]

device: cuda


# Data and model

The data and the network are the same as in Problem 1. The network has 3 hidden layers with 50 neurons.
It has no normalization layer.

In [ ]:
def load_mnist(train):
    ds = torchvision.datasets.MNIST("data", train=train, download=True)
    x = ds.data.float().div(255).unsqueeze(1)  # (n, 1, 28, 28)
    return x.to(device), ds.targets.to(device)


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), batch_size):
        j = idx[i:i + batch_size]
        yield x[j], y[j]


x_train, y_train = load_mnist(train=True)
x_test, y_test = load_mnist(train=False)
print(x_train.shape, x_test.shape)


def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 10),
    ).to(device)


def linear_layers(model):
    return [m for m in model if isinstance(m, nn.Linear)]

100%|██████████| 9.91M/9.91M [00:00<00:00, 54.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.67MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.82MB/s]


torch.Size([60000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


# Attacks

`pgd(model, x, y, eps, norm, steps)` is an untargeted PGD attack with a random start.
Use `norm="linf"` or `norm="l2"`. The function returns the adversarial images of the restart with the highest loss.

In [ ]:
def _project(delta, eps, norm):
    if norm == "linf":
        return delta.clamp(-eps, eps)
    n = delta.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
    return delta * (eps / n).clamp(max=1.0)


def pgd(model, x, y, eps, norm, steps=10, alpha=None, restarts=1):
    alpha = alpha or 2.5 * eps / steps
    best_x, best_loss = x.clone(), torch.full((len(x),), -float("inf"), device=x.device)
    for _ in range(restarts):
        if norm == "linf":
            delta = (torch.rand_like(x) * 2 - 1) * eps
        else:
            delta = torch.randn_like(x)
            r = torch.rand(len(x), 1, 1, 1, device=x.device) * eps
            delta = delta / delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * r
        delta = (x + delta).clamp(0, 1) - x
        for _ in range(steps):
            delta.requires_grad_(True)
            loss = F.cross_entropy(model(x + delta), y)
            grad, = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                if norm == "linf":
                    step = grad.sign()
                else:
                    step = grad / grad.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
                delta = _project(delta + alpha * step, eps, norm)
                delta = (x + delta).clamp(0, 1) - x
        with torch.no_grad():
            loss = F.cross_entropy(model(x + delta), y, reduction="none")
            better = loss > best_loss
            best_x[better], best_loss[better] = (x + delta)[better], loss[better]
    return best_x.detach()

# Verifier

A *region* has a center `c` (a batch of images), a radius `r` and a norm.
For $\ell_\infty$, the region is the box $[c - r, c + r] \cap [0, 1]^{784}$.
For $\ell_2$, the region is the ball $\{x' : \|x' - c\|_2 \leq r\}$. We do not intersect the ball with $[0, 1]^{784}$.
This is sound, but the bounds are a little looser.

* `ibp_margin_bounds(model, c, r, norm, y)` is IBP from Problem 1. For $\ell_2$, the bounds of the first linear layer
  are exact: $W_i c + b_i \pm r \|W_i\|_2$ for each neuron $i$. The last linear layer is merged with the specification.
* `crown_margin_bounds(model, c, r, norm, y)` is CROWN. At the input, $\min_{x'} a^\top x'$ over the $\ell_2$ ball is
  $a^\top c - r \|a\|_2$.
* `certified(model, x, y, norm)` uses the larger of the two lower bounds.

The two bound functions return a tensor `m` with the shape `(batch, 10)`.
The value `m[:, j]` is a lower bound of $f_y - f_j$ on the region, and `m[:, y] = 0`. Do not change the verifier.

In [ ]:
def first_layer_bounds(layer, c, r, norm):
    c = c.flatten(1)
    if norm == "linf":
        l, u = (c - r).clamp(0, 1), (c + r).clamp(0, 1)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
    else:
        mid, rad = layer(c), r * layer.weight.norm(dim=1)
    return mid - rad, mid + rad


def spec_matrix(y, n_classes=10):
    eye = torch.eye(n_classes, device=y.device)
    return eye[y].unsqueeze(1) - eye.unsqueeze(0)  # (batch, 10, 10): row j is e_y - e_j


def ibp_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
    l, u = F.relu(l), F.relu(u)
    last = layers[-1]
    w = last.weight[y].unsqueeze(1) - last.weight.unsqueeze(0)  # (batch, 10, n): W_y - W_j
    b = last.bias[y].unsqueeze(1) - last.bias.unsqueeze(0)
    mid, rad = (u + l) / 2, (u - l) / 2
    return (w @ mid.unsqueeze(2)).squeeze(2) - (w.abs() @ rad.unsqueeze(2)).squeeze(2) + b


def _relu_relaxation(l, u):
    active, unstable = l >= 0, (l < 0) & (u > 0)
    s = torch.where(active, torch.ones_like(l), torch.zeros_like(l))
    s = torch.where(unstable, u / (u - l).clamp(min=1e-12), s)
    c = torch.where(unstable, -s * l, torch.zeros_like(l))
    alpha = torch.where(active | (unstable & (u > -l)), torch.ones_like(l), torch.zeros_like(l))
    return alpha, s, c


def _backward_lower(A, layers, bounds, c0, r, norm):
    # Lower bound of A z, where z is the output of layers[-1] and bounds[i] are the bounds of the output of layers[i].
    const = torch.zeros(A.shape[:2], device=A.device)
    for i in range(len(layers) - 1, -1, -1):
        const = const + A @ layers[i].bias
        A = A @ layers[i].weight
        if i > 0:
            alpha, s, c = _relu_relaxation(*bounds[i - 1])
            a_pos, a_neg = A.clamp(min=0), A.clamp(max=0)
            const = const + (a_neg * c.unsqueeze(1)).sum(2)
            A = a_pos * alpha.unsqueeze(1) + a_neg * s.unsqueeze(1)
    c0 = c0.flatten(1)
    if norm == "linf":
        l0, u0 = (c0 - r).clamp(0, 1), (c0 + r).clamp(0, 1)
        return const + (A.clamp(min=0) * l0.unsqueeze(1)).sum(2) + (A.clamp(max=0) * u0.unsqueeze(1)).sum(2)
    return const + (A @ c0.unsqueeze(2)).squeeze(2) - r * A.norm(dim=2)


def crown_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    bounds = []
    for k in range(len(layers) - 1):
        eye = torch.eye(layers[k].out_features, device=c.device).expand(len(c), -1, -1)
        lo = _backward_lower(eye, layers[:k + 1], bounds, c, r, norm)
        hi = -_backward_lower(-eye, layers[:k + 1], bounds, c, r, norm)
        bounds.append((lo, hi))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


@torch.no_grad()
def certified(model, x, y, norm, eps=None):
    eps = EPS[norm] if eps is None else eps
    m = torch.maximum(ibp_margin_bounds(model, x, eps, norm, y), crown_margin_bounds(model, x, eps, norm, y))
    m[torch.arange(len(y)), y] = float("inf")
    return m.min(dim=1).values > 0

# Evaluation

`evaluate(model, name)` uses the 10,000 test images. It returns the clean accuracy, the certified accuracy for each
norm and the union certified accuracy. With `attacks=True`, it also returns the PGD accuracy (50 steps) for each norm
and for the union. The PGD accuracy is an upper bound of the true robust accuracy.
Use `evaluate` for all the numbers in your report.

In [ ]:
def evaluate(model, name, attacks=False, n=10000, batch_size=1000):
    model.eval()
    counts = {k: 0 for k in ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]}
    for x, y in batches(x_test[:n], y_test[:n], batch_size, False):
        with torch.no_grad():
            correct = model(x).argmax(1) == y
        cert = {p: certified(model, x, y, p) & correct for p in NORMS}
        counts["clean"] += correct.sum().item()
        counts["linf"] += cert["linf"].sum().item()
        counts["l2"] += cert["l2"].sum().item()
        counts["union"] += (cert["linf"] & cert["l2"]).sum().item()
        if attacks:
            rob = {}
            for p in NORMS:
                x_adv = pgd(model, x, y, EPS[p], p, steps=50)
                with torch.no_grad():
                    rob[p] = correct & (model(x_adv).argmax(1) == y)
                counts["pgd_" + p] += rob[p].sum().item()
            counts["pgd_union"] += (rob["linf"] & rob["l2"]).sum().item()
    res = {"name": name}
    res.update({k: 100 * v / n for k, v in counts.items() if attacks or not k.startswith("pgd")})
    return res


def print_results(rows):
    for r in rows:
        print(f"{r['name']:<26}" + "  ".join(f"{k} {v:6.2f}" for k, v in r.items()
                                     if k != "time" and isinstance(v, float)) +
              (f"  time/epoch {r['time']:5.1f} s" if "time" in r else ""))

# Training loop

For each batch, `train(model, loss_fn)` calls `loss_fn(model, x, y, t)` and does one optimizer step.
The value `t` is the training progress in epochs (a float). The schedule is the same as in Problem 1:

* For $t < 1$, `eps_scale(t)` is 0 (standard training).
* For $1 \leq t < 11$, `eps_scale(t)` increases linearly from 0 to 1.
* For $t \geq 11$, `eps_scale(t)` is 1.

A loss function uses the radius `eps_scale(t) * EPS[norm]`.

If you give the argument `gp_beta`, `train` uses `gradient_projection` (part (b)) when the radius is at its
final value ($t \geq 11$).

In [ ]:
WARMUP, RAMP = 1, 10
LR = 4e-3


def eps_scale(t):
    return 0.0 if t <= WARMUP else min((t - WARMUP) / RAMP, 1.0)


def train(model, loss_fn, epochs=20, lr=LR, batch_size=512, seed=0, gp_beta=None, verbose=True):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[15], gamma=0.1)
    params = [p for p in model.parameters() if p.requires_grad]
    n_batches = (len(x_train) + batch_size - 1) // batch_size
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for i, (x, y) in enumerate(batches(x_train, y_train, batch_size, True)):
            t = epoch + i / n_batches
            loss = loss_fn(model, x, y, t)
            opt.zero_grad()
            if gp_beta is not None and t >= WARMUP + RAMP:
                g_c = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, gn, gc in zip(params, g_n, g_c):
                    p.grad = gradient_projection(gn, gc, gp_beta)
            else:
                loss.backward()
            opt.step()
            total += loss.item()
        sched.step()
        if verbose:
            print(f"epoch {epoch + 1:2d}  loss {total / n_batches:.4f}  time {time.time() - start:.0f} s")
    return (time.time() - start) / epochs


def standard_loss(model, x, y, t):
    return F.cross_entropy(model(x), y)

# (a) Single-norm certified training

CURE uses the SABR loss from the lectures for each norm. SABR finds an adversarial example $x^*$ with PGD, and then it
calculates the IBP loss on a small ball of radius $\tau = \lambda \epsilon$ around $x^*$. PGD searches for $x^*$ in the
ball of radius $\epsilon - \tau$ around $x$, so that the small ball is inside the full ball.

1. `propagation_region`: Find a center $x^*$ with `pgd` (5 steps) in the ball of radius $\epsilon - \tau$ around $x$.
   Attack the model in eval mode. Then set the model back to train mode.
2. `certified_loss`: Calculate the cross-entropy of $\hat{z} = -\underline{m}$. Here, $\underline{m}$ are the IBP lower
   bounds of the margins on the small ball. Thus $\hat{z}_y = 0$.
   With `reduction="none"`, return the loss of each image.
3. `single_norm_loss`: For $t < 1$, return the standard loss. After that, return `certified_loss` with the radius
   `eps_scale(t) * EPS[norm]`.

The default ratios are `LAM = {"linf": 0.7, "l2": 0.7}`.

In [ ]:

LAM = {"linf": 0.7, "l2": 0.7}

def propagation_region(model, x, y, eps, lam, norm):
    tau = lam * eps
    search_radius = eps - tau

    was_training = model.training
    model.eval()

    try:
        if search_radius > 0:
            center = pgd(
                model, x, y,
                eps=search_radius,
                norm=norm,
                steps=5
            )
        else:
            center = x.detach().clone()
    finally:
        model.train(was_training)

    return center.detach(), tau


def certified_loss(
    model, x, y, eps, lam, norm,
    reduction="mean"
):
    center, tau = propagation_region(
        model, x, y, eps, lam, norm
    )

    margin_lb = ibp_margin_bounds(
        model, center, tau, norm, y
    )

    z_hat = -margin_lb

    return F.cross_entropy(
        z_hat, y, reduction=reduction
    )


def single_norm_loss(model, x, y, t, norm):
    if t < WARMUP:
        return standard_loss(model, x, y, t)

    eps = eps_scale(t) * EPS[norm]

    return certified_loss(
        model, x, y,
        eps=eps,
        lam=LAM[norm],
        norm=norm
    )


In [ ]:
models, results = {}, []
for name, loss_fn in [("standard", standard_loss),
                      ("linf", lambda m, x, y, t: single_norm_loss(m, x, y, t, "linf")),
                      ("l2", lambda m, x, y, t: single_norm_loss(m, x, y, t, "l2"))]:
    models[name] = make_model()
    sec = train(models[name], loss_fn)
    results.append(dict(evaluate(models[name], name, attacks=True), time=sec))
print_results(results)

epoch  1  loss 0.5812  time 1 s
epoch  2  loss 0.2138  time 1 s
epoch  3  loss 0.1522  time 1 s
epoch  4  loss 0.1289  time 2 s
epoch  5  loss 0.1063  time 2 s
epoch  6  loss 0.0958  time 2 s
epoch  7  loss 0.0782  time 2 s
epoch  8  loss 0.0700  time 2 s
epoch  9  loss 0.0629  time 3 s
epoch 10  loss 0.0549  time 3 s
epoch 11  loss 0.0547  time 3 s
epoch 12  loss 0.0463  time 3 s
epoch 13  loss 0.0428  time 4 s
epoch 14  loss 0.0385  time 4 s
epoch 15  loss 0.0383  time 4 s
epoch 16  loss 0.0207  time 4 s
epoch 17  loss 0.0155  time 5 s
epoch 18  loss 0.0145  time 5 s
epoch 19  loss 0.0134  time 5 s
epoch 20  loss 0.0127  time 5 s
epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4867  time 7 s
epoch  3  loss 0.4756  time 8 s
epoch  4  loss 0.4792  time 9 s
epoch  5  loss 0.4964  time 11 s
epoch  6  loss 0.5258  time 12 s
epoch  7  loss 0.5676  time 13 s
epoch  8  loss 0.6234  time 15 s
epoch  9  loss 0.6936  time 16 s
epoch 10  loss 0.7626  time 18 s
epoch 11  loss 0.8395  time 19 s
e

# (b) CURE

Let $\hat{z}^\infty$ and $\hat{z}^2$ be the negative IBP margin bounds on the two small balls of an image.
These are the same values as in `certified_loss`. Let $d^p = \mathrm{softmax}(\hat{z}^p)$. Complete `cure_loss`:
$$\mathcal{L}_{CURE} = \frac{1}{n} \sum_{i=1}^{n} \max\left(\mathcal{L}_{CE}(\hat{z}^\infty_i, y_i),\ \mathcal{L}_{CE}(\hat{z}^2_i, y_i)\right)
+ \eta \cdot \frac{1}{|\gamma|} \sum_{i \in \gamma} \mathrm{KL}\left(d^2_i \,\|\, d^\infty_i\right),$$
Here, $\gamma$ is the set of images in the batch that are certified for $\ell_2$ on their small ball.
For these images, all entries of $\hat{z}^2_i$ except $\hat{z}^2_{i, y_i}$ are negative.
The first term is CURE-Max. The second term is the bound alignment (BA).
For $t < 1$, return the standard loss. If $\gamma$ is empty, the second term is 0.

Then complete `gradient_projection` (GP). Let $g_c$ be the gradient of `cure_loss`, and $g_n$ the gradient of the
cross-entropy loss on the clean images. For each parameter tensor:
$$g = \beta \cdot \mathrm{GP}(g_n, g_c) + (1 - \beta) \cdot g_c, \qquad
\mathrm{GP}(g_n, g_c) = \begin{cases} \cos(g_n, g_c) \cdot g_n & \text{if } \cos(g_n, g_c) > 0 \\ 0 & \text{otherwise.} \end{cases}$$
This is the same operation as in RAMP.

This is a simplified version of CURE. The paper is different in three ways:

* It propagates an $\ell_\infty$ box also for $\ell_2$.
* It uses a different $\ell_2$ attack.
* It applies GP to the updates of a full epoch.

In [ ]:

def cure_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)

    scale = eps_scale(t)

    z = {}
    losses = {}

    for norm in NORMS:
        eps = scale * EPS[norm]
        center, tau = propagation_region(
            model, x, y, eps, LAM[norm], norm
        )

        margins = ibp_margin_bounds(
            model, center, tau, norm, y
        )

        z[norm] = -margins

        losses[norm] = F.cross_entropy(
            z[norm], y, reduction="none"
        )

    # CURE-Max: per-image maximum loss
    max_loss = torch.maximum(
        losses["linf"], losses["l2"]
    ).mean()

    if eta == 0:
        return max_loss

    # Images certified for L2 on the small ball
    z2_other = z["l2"].clone()
    z2_other.scatter_(1, y[:, None], -float("inf"))

    mask = z2_other.max(dim=1).values < 0

    if not mask.any().item():
        return max_loss

    # KL(d_L2 || d_Linf)
    log_p_inf = F.log_softmax(
        z["linf"][mask], dim=1
    )

    p_l2 = F.softmax(
        z["l2"][mask], dim=1
    )

    ba_loss = F.kl_div(
        log_p_inf,
        p_l2,
        reduction="batchmean"
    )

    return max_loss + eta * ba_loss


def gradient_projection(g_n, g_c, beta):
    gn = g_n.reshape(-1)
    gc = g_c.reshape(-1)

    dot = torch.dot(gn, gc)
    denominator = gn.norm() * gc.norm()

    cosine = dot / denominator.clamp(min=1e-12)

    if cosine.item() > 0:
        projected = cosine * g_n
    else:
        projected = torch.zeros_like(g_n)

    return beta * projected + (1 - beta) * g_c


In [ ]:
for name, loss_fn, beta in [("cure_max", lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0), None),
                            ("cure_max_ba", cure_loss, None),
                            ("cure_max_ba_gp", cure_loss, 0.5)]:
    models[name] = make_model()
    sec = train(models[name], loss_fn, gp_beta=beta)
    results.append(dict(evaluate(models[name], name, attacks=True), time=sec))
print_results(results)

epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4887  time 3 s
epoch  3  loss 0.4977  time 5 s
epoch  4  loss 0.5247  time 8 s
epoch  5  loss 0.5677  time 11 s
epoch  6  loss 0.6317  time 14 s
epoch  7  loss 0.7262  time 16 s
epoch  8  loss 0.8320  time 19 s
epoch  9  loss 0.9424  time 22 s
epoch 10  loss 1.0695  time 24 s
epoch 11  loss 1.2073  time 27 s
epoch 12  loss 1.2661  time 30 s
epoch 13  loss 1.2348  time 32 s
epoch 14  loss 1.2181  time 35 s
epoch 15  loss 1.2022  time 38 s
epoch 16  loss 1.1527  time 40 s
epoch 17  loss 1.1402  time 43 s
epoch 18  loss 1.1350  time 46 s
epoch 19  loss 1.1307  time 49 s
epoch 20  loss 1.1279  time 51 s
epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4881  time 3 s
epoch  3  loss 0.4963  time 6 s
epoch  4  loss 0.5235  time 9 s
epoch  5  loss 0.5661  time 12 s
epoch  6  loss 0.6276  time 14 s
epoch  7  loss 0.7220  time 17 s
epoch  8  loss 0.8306  time 20 s
epoch  9  loss 0.9502  time 23 s
epoch 10  loss 1.0789  time 26 s
epoch 11  loss 1.2

In [ ]:

import pandas as pd

cure_runs = []
cure_experiments = [
    ("cure_max", 0.0, None),
    ("cure_max_ba", 0.5, None),
    ("cure_max_ba_gp", 0.5, 0.5),
]

for name, eta, beta in cure_experiments:
    for seed in (0, 1):
        torch.manual_seed(seed)
        model = make_model()

        def loss_fn(m, x, y, t, eta=eta):
            return cure_loss(m, x, y, t, eta=eta)

        seconds = train(
            model, loss_fn,
            seed=seed, gp_beta=beta,
            verbose=False
        )

        result = evaluate(
            model, f"{name}_seed{seed}",
            attacks=True
        )

        result["method"] = name
        result["seed"] = seed
        result["time_per_epoch"] = seconds
        cure_runs.append(result)

        print(
            name, "seed", seed,
            "union", result["union"],
            "clean", result["clean"]
        )

df = pd.DataFrame(cure_runs)

mean_results = (
    df.groupby("method")
      .mean(numeric_only=True)
      .drop(columns="seed")
)

print("\n=== TWO-SEED MEANS ===")
print(mean_results.round(2).to_string())


cure_max seed 0 union 57.2 clean 92.15
cure_max seed 1 union 56.42 clean 92.43
cure_max_ba seed 0 union 57.87 clean 92.16
cure_max_ba seed 1 union 56.62 clean 92.07
cure_max_ba_gp seed 0 union 57.2 clean 92.69
cure_max_ba_gp seed 1 union 56.09 clean 92.26

=== TWO-SEED MEANS ===
                clean   linf     l2  union  pgd_linf  pgd_l2  pgd_union  time_per_epoch
method                                                                                 
cure_max        92.29  67.48  56.87  56.81     77.46   78.44      76.68            2.67
cure_max_ba     92.12  67.32  57.28  57.24     76.98   78.22      76.41            2.79
cure_max_ba_gp  92.48  67.03  56.72  56.64     76.74   78.30      76.16            2.90


# (c) Your method

Write the loss of your method in `my_loss`. Train it with `train` and the default settings, as for the other methods.
If your method needs a different update, write your own copy of `train`.
Do not change the verifier or `evaluate`.

In [ ]:
def joint_margin_loss(
    model, x, y, t,
    mu=0.5,
    use_margin=True
):
    if t < WARMUP:
        return standard_loss(model, x, y, t)

    z = {}
    margins = {}
    losses = {}

    for norm in NORMS:
        eps = eps_scale(t) * EPS[norm]

        center, tau = propagation_region(
            model, x, y,
            eps, LAM[norm], norm
        )

        m = ibp_margin_bounds(
            model, center, tau, norm, y
        )

        margins[norm] = m
        z[norm] = -m

        losses[norm] = F.cross_entropy(
            z[norm], y, reduction="none"
        )

    # Original CURE-Max
    max_loss = torch.maximum(
        losses["linf"],
        losses["l2"]
    ).mean()

    # Original Bound Alignment
    m_l2 = margins["l2"].clone()
    m_l2.scatter_(
        1, y[:, None], float("inf")
    )

    mask = m_l2.min(dim=1).values > 0

    if mask.any().item():
        ba_loss = F.kl_div(
            F.log_softmax(
                z["linf"][mask], dim=1
            ),
            F.softmax(
                z["l2"][mask], dim=1
            ),
            reduction="batchmean"
        )
    else:
        ba_loss = max_loss.new_zeros(())

    base_loss = max_loss + 0.5 * ba_loss

    # Ablation: original CURE-Max + BA
    if not use_margin:
        return base_loss

    # Minimum non-true-class margin per norm
    worst = []

    for norm in NORMS:
        m = margins[norm].clone()
        m.scatter_(
            1, y[:, None], float("inf")
        )

        worst.append(
            m.min(dim=1).values
        )

    # Worst margin across BOTH norms
    joint_margin = torch.minimum(
        worst[0], worst[1]
    )

    margin_penalty = F.softplus(
        -joint_margin
    ).mean()

    return base_loss + mu * margin_penalty


def my_loss(model, x, y, t):
    return joint_margin_loss(
        model, x, y, t,
        mu=0.5,
        use_margin=True
    )


In [ ]:
import pandas as pd
import numpy as np
import random

p2c_results = []
p2c_models = {}

experiments = [
    ("JW-CURE", True),
    ("Ablation_NoMargin", False)
]

for name, use_margin in experiments:
    for seed in [0, 1]:
        print(f"\n{name}, seed={seed}")

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        model = make_model()

        def loss_fn(m, x, y, t,
                    enabled=use_margin):
            return joint_margin_loss(
                m, x, y, t,
                mu=0.5,
                use_margin=enabled
            )

        sec = train(
            model,
            loss_fn,
            seed=seed,
            verbose=False
        )

        result = evaluate(
            model,
            f"{name}_seed{seed}",
            attacks=True
        )

        result["method"] = name
        result["seed"] = seed
        result["time_per_epoch"] = sec

        p2c_results.append(result)
        p2c_models[(name, seed)] = model

        print_results([{
            **result,
            "time": sec
        }])

df_p2c = pd.DataFrame(p2c_results)

p2c_means = (
    df_p2c
    .groupby("method")
    .mean(numeric_only=True)
    .drop(columns=["seed"])
)

print("\n=== P2(c) TWO-SEED MEANS ===")
print(p2c_means.round(2).to_string())



JW-CURE, seed=0
JW-CURE_seed0             clean  92.31  linf  67.77  l2  57.21  union  57.14  pgd_linf  77.48  pgd_l2  79.49  pgd_union  77.25  time_per_epoch   2.95  time/epoch   2.9 s

JW-CURE, seed=1
JW-CURE_seed1             clean  92.64  linf  66.46  l2  55.40  union  55.33  pgd_linf  77.85  pgd_l2  79.66  pgd_union  77.27  time_per_epoch   3.00  time/epoch   3.0 s

Ablation_NoMargin, seed=0
Ablation_NoMargin_seed0   clean  92.16  linf  67.78  l2  57.90  union  57.87  pgd_linf  77.37  pgd_l2  78.53  pgd_union  76.75  time_per_epoch   2.76  time/epoch   2.8 s

Ablation_NoMargin, seed=1
Ablation_NoMargin_seed1   clean  92.07  linf  66.86  l2  56.67  union  56.62  pgd_linf  76.59  pgd_l2  77.90  pgd_union  76.07  time_per_epoch   2.73  time/epoch   2.7 s

=== P2(c) TWO-SEED MEANS ===
                   clean   linf     l2  union  pgd_linf  pgd_l2  pgd_union  time_per_epoch
method                                                                                    
Ablation_NoMargin  9